In [ ]:
!pip install vaderSentiment transformers torch emoji -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import emoji
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from transformers import pipeline
import warnings
warnings.filterwarnings('ignore')

#TODO: fill in file name and drop the 3 columns that are not necessary
reviews_df = pd.read_csv("") # data name
reviews_df = reviews_df.drop(['', '', ''], axis=1) # after looking at the data see which 3 columns seem unecessary
reviews_df.head()

In [ ]:
# VADER
vader = SentimentIntensityAnalyzer()
compounds = [vader.polarity_scores(t)['compound'] for t in reviews_df['Review']]
reviews_df['vader_compound'] = compounds
reviews_df['vader_predicted_rating'] = [int(round((c + 1) * 2 + 1)) for c in compounds]
reviews_df

In [ ]:
# HuggingFace BERT
hf_pipe = pipeline("sentiment-analysis",
                   model="nlptown/bert-base-multilingual-uncased-sentiment",
                   device=0, truncation=True, max_length=512)

texts = [t if t.strip() else "neutral" for t in reviews_df['Review']]

results = []
for i in range(0, len(texts), 32):
    results.extend(hf_pipe(texts[i:i+32]))

reviews_df['hf_predicted_rating'] = [int(r['label'].split()[0]) for r in results]
reviews_df

In [ ]:
# accuracy results
vader_exact = (reviews_df['Rating'] == reviews_df['vader_predicted_rating']).mean() * 100
vader_off1 = (abs(reviews_df['Rating'] - reviews_df['vader_predicted_rating']) <= 1).mean() * 100
vader_diff = abs(reviews_df['Rating'] - reviews_df['vader_predicted_rating']).mean()

hf_exact = (reviews_df['Rating'] == reviews_df['hf_predicted_rating']).mean() * 100
hf_off1 = (abs(reviews_df['Rating'] - reviews_df['hf_predicted_rating']) <= 1).mean() * 100
hf_diff = abs(reviews_df['Rating'] - reviews_df['hf_predicted_rating']).mean()

pd.DataFrame({
    'Model': ['VADER', 'HuggingFace'],
    'Exact Match %': [vader_exact, hf_exact],
    'Off-by-One %': [vader_off1, hf_off1],
    'Avg Error': [vader_diff, hf_diff]
})

# **TO DO: Create Visualization 3**
Given the calculations for exact match%, off-by-one%, and avg error for with emojis create a visualization that will compare the two models. For example, you could create the same confusison matrices and bar graphs as the without_emojis code or you can choose your own such as a table to compare the data

In [ ]:
# visualization

In [ ]:
# examples where HuggingFace is better
sample_df = reviews_df[['Review', 'Rating', 'vader_predicted_rating', 'hf_predicted_rating']].copy()
sample_df['vader_error'] = abs(sample_df['Rating'] - sample_df['vader_predicted_rating'])
sample_df['hf_error'] = abs(sample_df['Rating'] - sample_df['hf_predicted_rating'])

sample_df[sample_df['hf_error'] < sample_df['vader_error']].head(5)

In [ ]:
# examples where VADER is better
sample_df[sample_df['vader_error'] < sample_df['hf_error']].head(5)

In [ ]:
# to show that it still has emojis
pd.set_option('display.max_colwidth', None)
reviews_df[['Review']].head(15)